In [151]:
import pandas as pd
listings = pd.read_csv('../data/listings.csv')

# 1)

In [152]:
listings['has_reviews'] = listings['number_of_reviews'] > 0
print((listings['has_reviews'] == False).mean().round(4))

0.1408


In [153]:
listings['no_reviews'] = listings['has_reviews'] == False
listings.groupby('neighbourhood_cleansed')['no_reviews'].mean().sort_values().round(4)

neighbourhood_cleansed
Södermalms                0.0760
Kungsholmens              0.0881
Skarpnäcks                0.1131
Hägersten-Liljeholmens    0.1336
Älvsjö                    0.1402
Östermalms                0.1414
Norrmalms                 0.1535
Spånga-Tensta             0.1910
Enskede-Årsta-Vantörs     0.1933
Bromma                    0.2064
Skärholmens               0.2083
Farsta                    0.2260
Rinkeby-Tensta            0.2593
Hässelby-Vällingby        0.2703
Name: no_reviews, dtype: float64

# 2)

In [154]:
reviewed = listings[listings['has_reviews']]
reviewed.shape
pd.qcut(reviewed['number_of_reviews'], 4).value_counts().sort_index()

number_of_reviews
(0.999, 4.0]      1010
(4.0, 13.0]       1032
(13.0, 36.0]       914
(36.0, 1293.0]     974
Name: count, dtype: int64

All the listings with 4 reviews go into same group. Same for 13 and 36. So groups will not be exactly same size.

# 3)

In [155]:
reviewed['groups'] = pd.qcut(reviewed['number_of_reviews'],
         4,
         labels=['0-4', '5-13', '14-36', '37-1293'],
         )
reviewed.groupby('groups')['review_scores_rating'].mean()

groups
0-4        4.793812
5-13       4.841928
14-36      4.831313
37-1293    4.803994
Name: review_scores_rating, dtype: float64

Review scores are slightly lower for the listings that have very few or very many reviews.
We can't really say for sure why, but we can speculate that listings that get a very bad review as the first review is less likely to be rented again. Listings that get reviewed a lot could possibly be very cheap, although we don't know that from the current data... but we could investigate it.

# 4)

In [ ]:
listings.groupby(['neighbourhood_cleansed','room_type']).agg(
    n_listings=('id', 'count'),
    mean_accommodates=('accommodates', 'mean'),
).sort_values('mean_accommodates', ascending=False).head(3)


,,n_listings,mean_accommodates
neighbourhood_cleansed,room_type,,
Östermalms,Shared room,4,9.000000
Kungsholmens,Shared room,8,6.500000
Spånga-Tensta,Entire home/apt,54,5.333333


In [207]:
grouped = listings.groupby(['neighbourhood_cleansed','room_type']).agg(
    n_listings=('id', 'count'),
    mean_accommodates=('accommodates', 'mean'),
).sort_values('mean_accommodates', ascending=False)
grouped30 = grouped[grouped['n_listings'] >= 30]
grouped30.sort_values('mean_accommodates', ascending=False).head(3)

,,n_listings,mean_accommodates
neighbourhood_cleansed,room_type,,
Spånga-Tensta,Entire home/apt,54,5.333333
Hässelby-Vällingby,Entire home/apt,118,5.245763
Farsta,Entire home/apt,145,4.965517


Östermalms/'Shared room' and Kungsholmens/'Shared room' have only 4 and 8 listings, so the mean of those two are less statistically significant than the other 3 values listed, which have far more listings.

# 5)

Grouped by number of bedrooms, how many listings are there with that bedroom count? and how many average beds and median beds does each group have?

In [217]:
listings.groupby('bedrooms').agg(
    n_listings=('id', 'count'),
    average_beds=('beds', 'mean'),
    median_beds=('beds', 'median')
).round(2)

,n_listings,average_beds,median_beds
bedrooms,,,
0.0,257,1.53,1.0
1.0,2490,1.44,1.0
2.0,739,2.56,2.0
3.0,588,3.61,3.0
4.0,351,4.64,4.0
5.0,116,5.85,5.0
6.0,23,7.17,7.0
7.0,3,9.00,9.0
8.0,1,12.00,12.0
